# House Price Prediction — Dataset and Preprocessing

This notebook loads the Ames Housing dataset, prepares the features, handles missing values, encodes categorical columns, and creates the train/test split.

**Dataset:** [Ames Housing Dataset on Kaggle](https://www.kaggle.com/datasets/prevek18/ames-housing-dataset/data)

If `AmesHousing.csv` is not already available in Colab, the next cell asks you to upload it.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

file_path = "/content/AmesHousing.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
else:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("Please upload AmesHousing.csv.")
    df = pd.read_csv(csv_files[0])

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = df.drop(columns=["Order", "PID"], errors="ignore")

if "SalePrice" not in df.columns:
    raise ValueError("The dataset must contain a 'SalePrice' column.")

X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Input features:", X.shape[1])
print("Numerical features:", len(num_cols))
print("Categorical features:", len(cat_cols))
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print(f"House price range: ${y.min():,.0f} to ${y.max():,.0f}")


## 1. Linear Regression Model

Train and evaluate the Linear Regression model using MAE, RMSE, and R². For MAE and RMSE, lower values are better; for R², higher is better.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results")
print("-------------------------")
print(f"MAE: ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")
print(f"R² Percentage: {r2 * 100:.2f}%")


In [ ]:
results = pd.DataFrame({
    "Actual Price": y_test.iloc[:5].values,
    "Predicted Price": y_pred[:5]
})
results["Predicted Price"] = results["Predicted Price"].round(2)
display(results)


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.title("Linear Regression: Actual vs Predicted House Prices")
plt.show()


In [ ]:
# Save the trained model and the preprocessing pipeline
joblib.dump(model, "house_price_model.pkl")
joblib.dump(preprocessor, "house_price_preprocessor.pkl")
print("Saved: house_price_model.pkl")
print("Saved: house_price_preprocessor.pkl")


In [ ]:
# Test a sample house from the held-out test set
sample_house = X_test.iloc[[0]]
sample_prediction = model.predict(preprocessor.transform(sample_house))[0]
actual_price = y_test.iloc[0]

print(f"Predicted house price: ${sample_prediction:,.2f}")
print(f"Actual house price:    ${actual_price:,.2f}")
print(f"Absolute error:        ${abs(sample_prediction - actual_price):,.2f}")


## Model files

After running all cells, `house_price_model.pkl` and `house_price_preprocessor.pkl` will be saved in the Colab session. Download them if you need local copies, or save them to Google Drive for persistence.


In [ ]:
# Optional: download the saved files to your computer
from google.colab import files
files.download("house_price_model.pkl")
files.download("house_price_preprocessor.pkl")
